# Counting · one → two

Build one mean-difference vector per decoder layer from 50 supplied QA pairs.
Sections 1–9 construct and save vectors only. Sections 10–12 are a separate,
optional denoising experiment. Use an A100 or L4 GPU and High-RAM.

## 1. Set up Colab

Run once in a fresh runtime. Restart the session after installation, then continue
with section 2. This branch must be published before cloning it.

In [ ]:
!nvidia-smi
!git clone --branch experiment/custom-steering-vectors --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1

## 2. Load the shared adapter

Run after restarting. Reuse this adapter for all subsequent sections.

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from google.colab import files
from IPython.display import Markdown, display
from PIL import Image

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter
from experiments.custom_utils import (
    prepare_pair, capture_pair, build_steering_vector,
    count_denoising_evaluations, steer_denoising,
)

SEED = 42
torch.manual_seed(SEED)
adapter = ShowO2Adapter()
OUTPUT = PROJECT / "outputs/counting/one_to_two"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(MODEL_ID, torch.cuda.get_device_name(0), adapter.dtype)

## 3. Load the 50 contrastive pairs

Positive sequences end in `two`; negative sequences end in `one`.
Both answers are supplied. Nothing is generated during vector construction.

In [ ]:
DATA = PROJECT / "inputs/counting/one_to_two/build_pairs.json"
pairs = json.loads(DATA.read_text(encoding="utf-8"))
layers = list(range(len(adapter.layers)))
print("Pairs:", len(pairs), "| Layers:", len(layers))
print("POSITIVE:\n", pairs[0]["positive"])
print("\nNEGATIVE:\n", pairs[0]["negative"])

## 4. Inspect the final extraction tokens

No padding, chat template, or EOS is appended. The selected position is the actual
last token, including only the final subtoken if an answer word is split.

In [ ]:
token_audit = []
for example in pairs:
    pair = prepare_pair(
        adapter, example["positive"], example["negative"], alignment="none",
    )
    for side, ids in pair["token_ids"].items():
        token_audit.append(dict(
            id=example["id"], side=side, position=len(ids) - 1,
            token_id=ids[-1], token=adapter.tokenizer.decode([ids[-1]]),
        ))

display(Markdown(
    "| Pair | Side | Position | Token ID | Final token |\n"
    "| --- | --- | --- | --- | --- |\n"
    + "".join(
        f"| {r['id']} | {r['side']} | {r['position']} | {r['token_id']} | {r['token']!r} |\n"
        for r in token_audit[:10]
    )
))

## 5. Extract the last-token states

Run two text-only forwards per pair. Capture block outputs before final norm.
Only the selected last-token states are retained across pairs, on CPU in FP32.

In [ ]:
positive_states = {layer: [] for layer in layers}
negative_states = {layer: [] for layer in layers}

for index, example in enumerate(pairs, 1):
    pair = prepare_pair(
        adapter, example["positive"], example["negative"], alignment="none",
    )
    captured = capture_pair(adapter, pair, layers=layers)
    pair_result = build_steering_vector(
        captured, tokens="last", token_reduction="mean",
        direction_method="difference", layer_reduction="none",
    )
    for layer in layers:
        positive_states[layer].append(pair_result["positive"][layer][0])
        negative_states[layer].append(pair_result["negative"][layer][0])
    print(f"{index}/{len(pairs)}: {example['id']}")

h_two = {layer: torch.stack(positive_states[layer]) for layer in layers}
h_one = {layer: torch.stack(negative_states[layer]) for layer in layers}
del captured, pair_result, positive_states, negative_states
print("Per-layer states:", tuple(h_two[layers[0]].shape))

## 6. Average pair differences independently per layer

`v[layer] = mean(h_two[layer] - h_one[layer], dim=0)`.
This equals the difference of the two group means, up to floating-point rounding.
No layer averaging, vector normalization, PCA, or steering coefficient is applied.

In [ ]:
differences = {layer: h_two[layer] - h_one[layer] for layer in layers}
vectors = {layer: differences[layer].mean(dim=0) for layer in layers}

for layer in layers:
    difference_of_means = h_two[layer].mean(0) - h_one[layer].mean(0)
    print(
        "Layer:", layer, "| shape:", tuple(vectors[layer].shape),
        "| norm:", vectors[layer].norm().item(),
        "| max rounding difference:", (vectors[layer] - difference_of_means).abs().max().item(),
    )

## 7. Inspect the operation

Six coordinates from one layer show the first pair's subtraction, followed by
the average over all pairs. Edit `preview_layer` to inspect another layer.

In [ ]:
preview_layer = layers[0]
rows = zip(
    h_two[preview_layer][0, :6].tolist(),
    h_one[preview_layer][0, :6].tolist(),
    differences[preview_layer][0, :6].tolist(),
)
display(Markdown(
    "| Coordinate | First h_two | First h_one | Difference |\n"
    "| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {p:.6f} | {n:.6f} | {d:.6f} |\n"
              for i, (p, n, d) in enumerate(rows))
))
sums = differences[preview_layer].sum(0)[:6]
rows = zip(
    sums.tolist(), (sums / len(pairs)).tolist(), vectors[preview_layer][:6].tolist(),
)
display(Markdown(
    "| Coordinate | Sum of pair differences | Sum / pairs | Saved vector |\n"
    "| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {s:.6f} | {m:.6f} | {v:.6f} |\n"
              for i, (s, m, v) in enumerate(rows))
))

## 8. Save the layer-specific vectors

Save the exact pairs, token audit, selected states, differences, and model/code
identifiers. Weight revisions remain the adapter's default Hugging Face revisions.

In [ ]:
metadata = dict(
    model=MODEL_ID, concept="counting", direction="one_to_two",
    n_pairs=len(pairs), alignment="none", tokens="last", layers=layers,
    method="mean(h_two - h_one)", normalization="none",
    capture="decoder block output before final norm",
    seed=SEED, dtype=str(adapter.dtype),
    project_revision=get_ipython().getoutput("git -C /content/show-o2-space rev-parse HEAD")[0],
    upstream_revision=get_ipython().getoutput("git -C /content/Show-o rev-parse HEAD")[0],
)
torch.save(dict(vectors=vectors, metadata=metadata), OUTPUT / "steering_vectors.pt")
torch.save(
    dict(positive=h_two, negative=h_one, differences=differences),
    OUTPUT / "last_token_states.pt",
)
(OUTPUT / "build_pairs.json").write_text(json.dumps(pairs, indent=2), encoding="utf-8")
(OUTPUT / "token_audit.json").write_text(json.dumps(token_audit, indent=2), encoding="utf-8")
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved:", OUTPUT)

## 9. Download the build results

Vector construction is complete. Sections below are an optional application pilot.

In [ ]:
archive = shutil.make_archive("/content/counting-build", "zip", OUTPUT)
files.download(archive)

## 10. Optional: generate a baseline and count evaluations

This is one application smoke test, not a held-out benchmark. It asks for one circle;
the intervention will attempt to change the count to two. Keep prompt, seed, steps,
and guidance fixed. Inspect whether the baseline actually contains one circle.

The counter measures real root-model evaluations. No interventions are installed.

In [ ]:
prompt = "One circle on a white background."
generation_kwargs = dict(steps=50, guidance=5.0, seed=SEED)

with count_denoising_evaluations(adapter) as baseline_trace:
    baseline = adapter.generate(prompt, **generation_kwargs)

total_evaluations = baseline_trace["evaluations"]
print("Measured evaluations:", total_evaluations)

## 11. Optional: apply the vectors during denoising

Each selected layer receives its own vector at the block output. By default, add
to all visual positions, at every measured evaluation, on conditional CFG row 0.
Prompt text, time, special tokens, padding, and the unconditional row stay untouched.

Selections accept indices, lists, slices, `"first"`, `"last"`, and `"all"`.
`iterations=slice(0, 24)` means the first 24 evaluations; indices are zero-based.
Text selections refer to prompt content only; this sampler generates no AR text.

`operation="add"`: `h + alpha*v`.
`operation="component"`: `h + alpha*(h @ unit_v)*unit_v`;
`alpha=-1` removes the component and `alpha=1` doubles it.
A callable uses `operation(hidden, vector, *, alpha)` and returns edited states.

To edit text independently, replace `text_rule=None` with
`dict(tokens="all", operation="add", alpha=0.1)`.
Layer output hooks are removed when the context ends.

In [ ]:
application_layers = "all"
iterations = "all"
text_rule = None
visual_rule = dict(tokens="all", operation="add", alpha=0.1)

with steer_denoising(
    adapter, vectors=vectors, total_evaluations=total_evaluations,
    layers=application_layers, iterations=iterations,
    text=text_rule, visual=visual_rule,
) as steering_trace:
    steered = adapter.generate(prompt, **generation_kwargs)


### Inspect the numerical edit

Optional preview of six coordinates from the first edited token. Run or skip this
cell independently; it does not generate images or save files.


In [ ]:
preview_key = next(iter(steering_trace["preview"]))
preview = steering_trace["preview"][preview_key]
rows = zip(
    preview["before"].tolist(), preview["after"].tolist(),
    (preview["after"] - preview["before"]).tolist(),
)
print("Layer:part:", preview_key, "| iteration:", preview["iteration"], "| position:", preview["position"])
display(Markdown(
    "| Coordinate | Before | After | Change |\n| --- | --- | --- | --- |\n"
    + "".join(f"| {i} | {b:.6f} | {a:.6f} | {d:.6f} |\n"
              for i, (b, a, d) in enumerate(rows))
))


### Compare images

Left: baseline. Right: steering with the same prompt, seed, and sampling settings.


In [ ]:
width, height = baseline.size
comparison = Image.new("RGB", (width * 2, height))
comparison.paste(baseline, (0, 0))
comparison.paste(steered, (width, 0))
print("Left: baseline | Right: steered")
print("Evaluations:", steering_trace["evaluations"])
print("Layer/modality injections:", len(steering_trace["injections"]))
display(comparison)

## 12. Optional: save and download

Save images, application settings, and the intervention trace. No generation or
numerical preview is run here.


In [ ]:
baseline.save(OUTPUT / "baseline.png")
steered.save(OUTPUT / "steered.png")
comparison.save(OUTPUT / "comparison.png")
torch.save(
    dict(
        prompt=prompt, generation=generation_kwargs, layers=application_layers,
        iterations=iterations, text=text_rule, visual=visual_rule,
        trace=steering_trace, vector_metadata=metadata,
    ),
    OUTPUT / "application.pt",
)
archive = shutil.make_archive("/content/counting-application", "zip", OUTPUT)
files.download(archive)